# Plot zonal figs of iSnobal comparisons with SNOW-17 products

In [ ]:
import sys

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
import xarray as xr
import seaborn as sns
import copy

from pathlib import PurePath

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/env/')
import helpers as h

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/ucrb-isnobal/scripts/')
import processing as proc

# Set seaborn palette
sns.set_palette('icefire')

In [ ]:
%load_ext autoreload
%autoreload 2

### iSnobal zonal data

In [ ]:
def extract_runtype_zonal_basin_gdf(zonal_basin_data_fns, runtype='baseline'):
    # Read in the zonal basin data files and separate by runtype
    df_list = [gpd.read_file(zonal_fn) for zonal_fn in zonal_basin_data_fns if runtype in zonal_fn]

    # Convert all data columns to meters
    df_list = [convert_to_meters(df) for df in df_list]

    # Set the zone column as the index for each dataframe
    df_list = [df.set_index('zone') for df in df_list]

    # Remove the zonal_classification and geometry columns from all but the first dataframe to avoid duplicate colnames
    for i in range(1, len(df_list)):
        df_list[i] = df_list[i].drop(columns=['zonal_classification', 'geometry'])

    # Join the dataframes into a single big df for each runtype, based on zone name
    gdf = pd.concat(df_list, axis=1)
    return gdf

def convert_to_meters(zonal_basin_data):
    '''Convert to meters rather than equivalent to mm SWE for all columns
    that are not zone, zonal_classification or geometry'''
    for col in zonal_basin_data.columns:
        if col not in ['zone', 'zonal_classification', 'geometry']:
            zonal_basin_data[col] = zonal_basin_data[col] / 1000.0  # Convert from mm to m
    return zonal_basin_data

def convert_basin_gdf_to_zonal_por_df(gdf, runtype='baseline'):
    '''Convert the basin gdfs into a big zonal period of record df'''
    # Turn the zone index into a column
    gdf.reset_index(inplace=True)

    # Transpose the dataframe
    df = gdf.set_index('zone').T

    # Drop the zonal classification row and the geometry row
    df = df.drop(index='zonal_classification')
    df = df.drop(index='geometry')

    # Create a new date column based on the index values and coerce to new format %Y-%m-%d
    df['date'] = pd.to_datetime([f'{dt[:4]}-{dt[4:6]}-{dt[-2:]}' for dt in [f.split('_')[1] for f in df.index]], format='%Y-%m-%d')

    # Reset the date column as the new index column
    df.set_index('date', inplace=True)

    # And reset the index to a regular column with no label
    # setting date as a regular column so the df is a single level
    df.reset_index(inplace=True)

    # Rename the name of the columns to the runtype
    df.columns.name = runtype
    return df


In [ ]:
# Read in basin data
basin = 'yampa'
var = 'specific_mass'
data_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/ancillary_sdswe_products/CBRFC_SNOW17'

# Get list of zonal basin isnobal data filenames
zonal_basin_data_fns = h.fn_list(data_dir, f'runtype_zonal/*{basin}*{var}*gpkg')

# Extract the gdf by runtype
baseline_gdf = extract_runtype_zonal_basin_gdf(zonal_basin_data_fns, runtype='baseline')
hrrrspires_gdf = extract_runtype_zonal_basin_gdf(zonal_basin_data_fns, runtype='hrrrspires')

# Convert the gdfs to zonal period of record dataframes
baseline_df = convert_basin_gdf_to_zonal_por_df(baseline_gdf, runtype='baseline')
hrrrspires_df = convert_basin_gdf_to_zonal_por_df(hrrrspires_gdf, runtype='hrrrspires')

In [ ]:
baseline_gdf.head()

In [ ]:
baseline_df.head()

In [ ]:
baseline_df.plot(x='date')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small');

### Zone nomenclature

In [ ]:
# Grab the zone names from these dataframes
zones = baseline_df.columns[1:]

# Trim the last three characters to match the zone names in the csv files
zones_abbrev = np.unique([zone[:-3] for zone in zones])
print(zones)
print(zones_abbrev)

### Load SNOW17 data as POR files

In [ ]:
# Read in SNOW17 data csvs only if csv name contains one of the zone name abbreviations
basin_csv_fns = []
for zone in zones_abbrev:
    zone_csv = h.fn_list(data_dir, f'csvs/{zone}*csv')[0]
    basin_csv_fns.append(zone_csv)

df_list = [pd.read_csv(csv, index_col=0) for csv in basin_csv_fns]
for df in df_list:
    print(df.shape)

df_list[0]

In [ ]:
def process_cbrfc_df(in_df, WY=None):
    '''
    Generate date column and filter to specified water year (WY)
    in_df: pandas DataFrame with columns 'cal_yr', 'mon', and 'zday'
    WY: Water year to filter the DataFrame to, e.g., 2022 for WY 2022
    '''
    df = copy.deepcopy(in_df)
    # Generate a date column from the cal_yr, mon and zday columns by using a lambda function to do row-wise conversion
    df['date'] = pd.to_datetime(df.apply(lambda row: f"{row['cal_yr']}-{row['mon']:02d}-{row['zday']:02d}", axis=1))

    # Drop the cal_yr, mon, and zday columns
    df.drop(columns=['cal_yr', 'mon', 'zday'], inplace=True)
    # Trim to the specified water year Oct 1 through September 30 if WY input detected
    if WY is not None:
        df = df[(df['date'] >= f'{WY - 1}-10-01') & (df['date'] <= f'{WY}-09-30')]
    return df

# Process each zonal df in the list
df_list = [process_cbrfc_df(df) for df in df_list]
df_list[0]

In [ ]:
def recompile_snow17_df_list(df_list):
    # Now recompile the snow17 dataframe list into a single one by moving the opid and appending to the swe column
    snow17_df = pd.DataFrame()

    # For each unique "subzone" in opid column, split into new df
    # These should remain in zone order as above since deriving
    # zone names from np.unique
    for df in df_list:
        subzones = np.unique(df['opid'])
        for subzone in subzones:
            sub_df = df[df['opid'] == subzone]
            # Append subzone name to the swe column
            sub_df = sub_df.rename(columns={'swe': f'{subzone}'})
            # Drop opid column
            sub_df.drop(columns=['opid'], inplace=True)
            print(subzone, sub_df.shape)
            # Convert date column to index
            sub_df.set_index('date', inplace=True)

            # Concatenate to the larger snow17_df DataFrame
            snow17_df = pd.concat([snow17_df, sub_df], axis=1)

    # Convert from inches to meters
    snow17_df = snow17_df / 39.37

    # Reset the index
    snow17_df = snow17_df.reset_index()

    # Rename the name of the columns to snow17
    snow17_df.columns.name = 'snow17'

    return snow17_df

snow17_df = recompile_snow17_df_list(df_list)

In [ ]:
# Set legend to be outside the plot
snow17_df.plot(x='date')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small');

## Plot these on top of each other

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(16, 5))
snow17_df.plot(x='date', ax=ax, linestyle=':', linewidth=0.65)
baseline_df.plot(x='date', ax=ax, linestyle='-.', linewidth=1)
hrrrspires_df.plot(x='date', ax=ax, linestyle='--', linewidth=0.5)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small');

### Break this up by zone

In [ ]:
big_df_list = [snow17_df, baseline_df, hrrrspires_df]
big_df_list_names = ['snow17', 'baseline', 'hrrrspires']
zone_dfs = {}
for zone in zones_abbrev:
    print(zone)
    # Extract the columns which contain a snippet of this zone
    sub_df_list =[pd.concat([df['date']]+[df[col] for col in df.columns if zone in col], axis=1) for df in big_df_list]
    # # Retain the name of the columns to minimize confusion
    # for jdx, df in enumerate(sub_df_list):
    #     df.columns.name = big_df_list_names[jdx]

    # Or append the name to each column
    for jdx, df in enumerate(sub_df_list):
        sub_df_list[jdx].columns = [f'{col}_{big_df_list_names[jdx]}' for col in df.columns]
        # Rename the date column to just 'date'
        sub_df_list[jdx].rename(columns={f'date_{big_df_list_names[jdx]}': 'date'}, inplace=True)
        # Set the index to the date column
        sub_df_list[jdx].set_index('date', inplace=True)

    # Concatenate the sub_df_list into a single df based on the date column
    zone_df = pd.concat(sub_df_list, axis=1, ignore_index=False)

    # Set the name of the columns to the zone name
    zone_df.columns.name = zone

    # Store in the dict
    zone_dfs[zone] = zone_df

zone_df

In [ ]:
zone_df.plot()
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small');

In [ ]:
# Update styling
# LF is yellow
# MF is seagreen
# UF is purple
# HRRR-SPIReS linestyle is -. (dashdot))
# CBRFC SNOW17 linestyle is - (solid)

def identify_styles(col, zone, verbose=True):
    '''Based on the column name, break down the relevant components and get appropriate styles'''
    # Assign line color based on subzone
    subzone = col.split(f'{zone}')[1].split('_')[0][-2:]
    if verbose:
        print(subzone)
    subzone_dict = {'LF': 'gold', 'MF': 'seagreen', 'UF': 'purple'}
    try:
        color = subzone_dict[subzone]
    except KeyError:
        print(f'Unknown subzone: {subzone}')
        # assign default values
        color = 'black'

    # Assign linestyle based on runtype
    runtype = col.split(f'{subzone}_')[1]
    if verbose:
        print(runtype)
    runtype_dict = {'baseline': ((0, (1, 2)), 1.2, 0.8), 'hrrrspires': ('-.', 2, 1),
                    'snow17': ('-', 4, 0.25), 'ua': (':', 1, 0.8), 'nwm': ('--', 0.8, 1)}
    try:
        linestyle = runtype_dict[runtype][0]
        linewidth = runtype_dict[runtype][1]
        alpha = runtype_dict[runtype][2]
    except KeyError:
        print(f'Unknown runtype: {runtype}')
        # assign default values
        linestyle = '--'
        linewidth = 2
        alpha = 1

    return color, linestyle, linewidth, alpha

In [ ]:
# # Plot each column individually
# for zone in zones_abbrev:
#     # print(zone)
#     fig, ax = plt.subplots(1, 1, figsize=(16, 5))
#     # Extract the df for this zone
#     zone_df = zone_dfs[zone]
#     for col in zone_df.columns:
#         # print(col)
#         color, linestyle, linewidth, alpha = identify_styles(col=col, zone=zone, verbose=False)
#         zone_df[col].plot(ax=ax, color=color, linestyle=linestyle, linewidth=linewidth, alpha=alpha, label=col)
#     ax.set_xlabel('')
#     ax.set_ylabel('SWE [m]')
#     ax.set_ylim(0, 1.25)
#     ax.grid('grey', linestyle='--', linewidth=0.5)
#     ax.annotate(f'{basin.capitalize()} ({zone})', xy=(0.95, 0.9), xycoords='axes fraction', ha='right', fontsize='large', fontstyle='oblique')
#     # Turn off box outline
#     plt.legend(bbox_to_anchor=(1.025, 1), loc='upper left', fontsize='medium', frameon=False)
#     # Save the figure
#     fig_fn = PurePath(data_dir, f'basin_zonal_por_figures/{basin}_{zone}_isnobal_snow17_por_swe_comparison.png')
#     plt.savefig(fig_fn, bbox_inches='tight', dpi=300)
#     print(fig_fn)

In [ ]:
# # Plot everything again without baseline
# # Plot each column individually
# for zone in zones_abbrev:
#     # print(zone)
#     fig, ax = plt.subplots(1, 1, figsize=(16, 5))
#     # Extract the df for this zone
#     zone_df = zone_dfs[zone]
#     for col in zone_df.columns:
#         if 'baseline' in col:
#             continue
#         else:
#             color, linestyle, linewidth, alpha = identify_styles(col=col, zone=zone, verbose=False)
#             zone_df[col].plot(ax=ax, color=color, linestyle=linestyle, linewidth=linewidth, alpha=alpha, label=col)
#     ax.set_xlabel('')
#     ax.set_ylabel('SWE [m]')
#     ax.set_ylim(0, 1.25)
#     ax.grid('grey', linestyle='--', linewidth=0.5)
#     ax.annotate(f'{basin.capitalize()} ({zone})', xy=(0.95, 0.9), xycoords='axes fraction', ha='right', fontsize='large', fontstyle='oblique')
#     # Turn off box outline
#     plt.legend(bbox_to_anchor=(1.025, 1), loc='upper left', fontsize='medium', frameon=False)
#     # Save the figure
#     fig_fn = PurePath(data_dir, f'basin_zonal_por_figures/{basin}_{zone}_isnobal_snow17_por_swe_comparison_nobaseline.png')
#     plt.savefig(fig_fn, bbox_inches='tight', dpi=300)
#     print(fig_fn)

# Try to work in UA and NWM (for which you only have point extracts for water years or day extracts for the basin)

### UA data

In [ ]:
# Read in the data for the designated time period on the fly and form a data cube
def generate_yyyymmdd(WY):
    '''Generates a list of YYYYMMDD strings for the input water year WY'''
    date_list = []

    # Begin with October 1 of the previous year and end with Sept 30 of the current year
    start_date = f'{WY - 1}1001'
    end_date = f'{WY}0930'
    # Generate the strings of dates by incrementing the day
    current_date = start_date
    while current_date <= end_date:
        date_list.append(current_date)
        # Increment the day
        current_date = pd.to_datetime(current_date, format='%Y%m%d') + pd.Timedelta(days=1)
        current_date = current_date.strftime('%Y%m%d')
    return date_list

In [ ]:
import os
from fsspec.implementations.http import HTTPFileSystem
# Output this to file for the basin, variable, and water year
UA_var = 'SWE'
ua_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/ancillary_sdswe_products/UA'

for WY in [2021, 2022, 2023, 2024]:
    print(WY)
    out_fn = f'{ua_dir}/{basin}/{basin}_ua_800m_wy{WY}_{UA_var}.nc'
    # if the data cube (netcdf) does not exist, make it!
    if os.path.exists(out_fn):
        print(f'Output file {out_fn} already exists, skipping...')
    else:
        # Generate the list of dates for the water year
        date_list = generate_yyyymmdd(WY)
        # Create the list of URLs for the UA SWE data
        ua_url_list = [f'https://climate.arizona.edu/data/UA_SWE/DailyData_800m/WY{WY}/UA_SWE_Depth_800m_v1_{dt}_stable.nc' for dt in date_list]
        print(ua_url_list[0])

        # Modified from https://github.com/pydata/xarray/issues/3653#issuecomment-570156277
        # and from spatial_comps_density_SWE.ipynb
        fs = HTTPFileSystem()
        fobj_list = []
        for url in ua_url_list:
            try:
                fobj_list.append(fs.open(url))
            except:
                print(f'Error opening URL: {url}')
                break
        # fobj_list = [fs.open(url) for url in ua_url_list]
        ua_ds = xr.open_mfdataset(fobj_list, engine='h5netcdf', drop_variables=['DEPTH', 'time_str'])  # Drop SWE and time_str variables
        # Reassign coordinate names
        ua_ds = ua_ds.rename({'lat': 'y', 'lon': 'x'})
        # Explicitly write out the crs
        ua_ds.rio.write_crs(input_crs=ua_ds.crs.attrs['spatial_ref'], inplace=True)
        # Convert to m (currently in millimeters) - don't do this, leave in mm for now, there is handling for this up above!
        # ua_ds[UA_var] = ua_ds[UA_var] / 1000
        # Note the units in the file
        ua_ds[UA_var].attrs['units'] = 'mm'
        # Reproject the geodataframe into the same CRS as the UA dataset
        hrrrspires_gdf_reproj = hrrrspires_gdf.to_crs(crs=ua_ds.rio.crs)
        # Do some spatial slicing using the geometry within the hrrrspires gdf
        bbox = hrrrspires_gdf_reproj.total_bounds
        # Write the CRS to the dataset, this doesn't seem to stick from above
        ua_ds.rio.write_crs(input_crs=ua_ds.rio.crs, inplace=True)
        # Clip the dataset to the bounding box of the hrrrspires gdf
        ua_sliced = ua_ds.rio.clip_box(minx=bbox[0], miny=bbox[1], maxx=bbox[2], maxy=bbox[3])
        # Save to netcdf
        ua_sliced.to_netcdf(out_fn, mode='w', format='NETCDF4', engine='h5netcdf')

In [ ]:
# Pause and make sure calc_cbrfc script has been run
# time for f in 2021 2022 2023 2024 ; do calc_cbrfc_zonalmean.py blue $f -r ua -var SWE ; done

In [ ]:
# Extract the gdf by runtype
zonal_basin_data_fns = h.fn_list(data_dir, f'runtype_zonal/*{basin}*{UA_var}*gpkg')
ua_gdf = extract_runtype_zonal_basin_gdf(zonal_basin_data_fns, runtype='ua')

# Convert the gdfs to zonal period of record dataframes
ua_df = convert_basin_gdf_to_zonal_por_df(ua_gdf, runtype='ua')
ua_df.plot(x='date')

In [ ]:
big_df_list = [snow17_df, baseline_df, hrrrspires_df, ua_df]
big_df_list_names = ['snow17', 'baseline', 'hrrrspires', 'ua']
zone_dfs = {}
for zone in zones_abbrev:
    print(zone)
    # Extract the columns which contain a snippet of this zone
    sub_df_list =[pd.concat([df['date']]+[df[col] for col in df.columns if zone in col], axis=1) for df in big_df_list]
    # # Retain the name of the columns to minimize confusion
    # for jdx, df in enumerate(sub_df_list):
    #     df.columns.name = big_df_list_names[jdx]

    # Or append the name to each column
    for jdx, df in enumerate(sub_df_list):
        sub_df_list[jdx].columns = [f'{col}_{big_df_list_names[jdx]}' for col in df.columns]
        # Rename the date column to just 'date'
        sub_df_list[jdx].rename(columns={f'date_{big_df_list_names[jdx]}': 'date'}, inplace=True)
        # Set the index to the date column
        sub_df_list[jdx].set_index('date', inplace=True)

    # Concatenate the sub_df_list into a single df based on the date column
    zone_df = pd.concat(sub_df_list, axis=1, ignore_index=False)

    # Set the name of the columns to the zone name
    zone_df.columns.name = zone

    # Store in the dict
    zone_dfs[zone] = zone_df

zone_df

In [ ]:
# # Plot everything again without baseline
# # Plot each column individually
# for zone in zones_abbrev:
#     # print(zone)
#     fig, ax = plt.subplots(1, 1, figsize=(16, 5))
#     # Extract the df for this zone
#     zone_df = zone_dfs[zone]
#     for col in zone_df.columns:
#         if 'baseline' in col:
#             continue
#         else:
#             color, linestyle, linewidth, alpha = identify_styles(col=col, zone=zone, verbose=False)
#             zone_df[col].plot(ax=ax, color=color, linestyle=linestyle, linewidth=linewidth, alpha=alpha, label=col)
#     ax.set_xlabel('')
#     ax.set_ylabel('SWE [m]')
#     ax.set_ylim(0, 1.25)
#     ax.grid('grey', linestyle='--', linewidth=0.5)
#     ax.annotate(f'{basin.capitalize()} ({zone})', xy=(0.95, 0.9), xycoords='axes fraction', ha='right', fontsize='large', fontstyle='oblique')
#     # Turn off box outline
#     plt.legend(bbox_to_anchor=(1.025, 1), loc='upper left', fontsize='medium', frameon=False)
#     # Save the figure
#     fig_fn = PurePath(data_dir, f'basin_zonal_por_figures/{basin}_{zone}_isnobal_snow17_por_swe_comparison_nobaseline_ua.png')
#     plt.savefig(fig_fn, bbox_inches='tight', dpi=300)
#     print(fig_fn)

### NWM data

In [ ]:
from s3fs import S3FileSystem, S3Map
# Output this to file for the basin, variable, and water year
NWM_var = 'SNEQV'
nwm_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/ancillary_sdswe_products/NWM'

# Read in NWM proj4 string
proj_fn = "/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/ancillary/NWM_datasets_proj4.txt"
with open(proj_fn, "r") as f:
    proj4 = f.read()
    print(proj4)

for WY in [2021, 2022]:
    print(WY)
    out_fn = f'{nwm_dir}/{basin}/{basin}_nwm_wy{WY}_{NWM_var}.nc'
    # if the data cube (netcdf) does not exist, make it!
    if os.path.exists(out_fn):
        print(f'Output file {out_fn} already exists, skipping...')
    else:
        # As of March 2025, NWM version 3 only runs till Jan 2023, so no full WY 2023 or 2024!
        # https://registry.opendata.aws/nwm-archive/
        bucket = 's3://noaa-nwm-retrospective-3-0-pds/CONUS/zarr'
        fs = S3FileSystem(anon=True)
        ds = xr.open_dataset(S3Map(f"{bucket}/ldasout.zarr", s3=fs), engine='zarr')

        # Slice time from NWM spatial crop for this water year
        nwm_ds = ds[NWM_var].sel(time=slice(f'{WY-1}-10-01', f'{WY}-09-30'))

        # Note the units in the file
        nwm_ds.attrs['units'] = 'mm'

        # Reproject the geodataframe into the same CRS as the NWM dataset
        hrrrspires_gdf_reproj = hrrrspires_gdf.to_crs(crs=proj4)
        # Do some spatial slicing using the geometry within the hrrrspires gdf
        bbox = hrrrspires_gdf_reproj.total_bounds
        # Write the CRS to the dataset, this doesn't seem to stick from above
        nwm_ds.rio.write_crs(input_crs=proj4, inplace=True)

        # Clip the dataset to the bounding box of the hrrrspires gdf
        nwm_sliced = nwm_ds.rio.clip_box(minx=bbox[0], miny=bbox[1], maxx=bbox[2], maxy=bbox[3])

        # Save to netcdf (these should contain 8 timesteps per day, or 2920 timesteps per year)
        nwm_sliced.to_netcdf(out_fn, mode='w', format='NETCDF4', engine='h5netcdf')
        print(f'Saved NWM data to {out_fn}')
        del ds

In [ ]:
# Pause and make sure calc_cbrfc script has been run
# time for f in 2021 2022 2023 2024 ; do calc_cbrfc_zonalmean.py blue $f -r nwm -var SNEQV ; done

In [ ]:
# Extract the gdf by runtype
zonal_basin_data_fns = h.fn_list(data_dir, f'runtype_zonal/*{basin}*{NWM_var}*gpkg')
nwm_gdf = extract_runtype_zonal_basin_gdf(zonal_basin_data_fns, runtype='nwm')

# Convert the gdfs to zonal period of record dataframes
nwm_df = convert_basin_gdf_to_zonal_por_df(nwm_gdf, runtype='nwm')
nwm_df.plot(x='date')

In [ ]:
big_df_list = [snow17_df, baseline_df, hrrrspires_df, ua_df, nwm_df]
big_df_list_names = ['snow17', 'baseline', 'hrrrspires', 'ua', 'nwm']
zone_dfs = {}
for zone in zones_abbrev:
    print(zone)
    # Extract the columns which contain a snippet of this zone
    sub_df_list =[pd.concat([df['date']]+[df[col] for col in df.columns if zone in col], axis=1) for df in big_df_list]

    # Or append the name to each column
    for jdx, df in enumerate(sub_df_list):
        sub_df_list[jdx].columns = [f'{col}_{big_df_list_names[jdx]}' for col in df.columns]
        # Rename the date column to just 'date'
        sub_df_list[jdx].rename(columns={f'date_{big_df_list_names[jdx]}': 'date'}, inplace=True)
        # Set the index to the date column
        sub_df_list[jdx].set_index('date', inplace=True)

    # Concatenate the sub_df_list into a single df based on the date column
    zone_df = pd.concat(sub_df_list, axis=1, ignore_index=False)

    # Set the name of the columns to the zone name
    zone_df.columns.name = zone

    # Store in the dict
    zone_dfs[zone] = zone_df

zone_df

In [ ]:
# Update the font size parameters
plt.rcParams.update({'font.size': 20,
                     'axes.labelsize': 22,
                     'axes.titlesize': 18,
                     'xtick.labelsize': 20,
                     'ytick.labelsize': 20,
                     'legend.fontsize': 18})

In [ ]:
# Plot each column individually
isnobal_only = False
ua_off = False
nwm_off = False
# Plot isnobal (baseline and HRRR-SPIReS only)
if isnobal_only:
    fig_fn = PurePath(data_dir, f'basin_zonal_por_figures/{basin}_{zone}_isnobal_snow17_por_swe_comparison.png')
    nobaseline = False
else:
    # Otherwise do not plot baseline
    nobaseline = True
    if not ua_off and nwm_off:
        # if ua_off is False, plot UA and HRRR-SPIReS and SNOW17
        fig_fn = PurePath(data_dir, f'basin_zonal_por_figures/{basin}_{zone}_isnobal_snow17_por_swe_comparison_nobaseline_ua.png')
    elif not nwm_off and ua_off:
        # if nwm_off is False, plot NWM and HRRR-SPIReS and SNOW17
        fig_fn = PurePath(data_dir, f'basin_zonal_por_figures/{basin}_{zone}_isnobal_snow17_por_swe_comparison_nobaseline_nwm.png')
    elif not nwm_off and not ua_off:
        # if both nwm_off and ua_off are False, plot NWM, UA, HRRR-SPIReS and SNOW17
        fig_fn = PurePath(data_dir, f'basin_zonal_por_figures/{basin}_{zone}_isnobal_snow17_por_swe_comparison_nobaseline_ua_nwm.png')
    else:
        # only plot HRRR-SPIReS and SNOW17
        fig_fn = PurePath(data_dir, f'basin_zonal_por_figures/{basin}_{zone}_isnobal_snow17_por_swe_comparison_nobaseline.png')

for zone in zones_abbrev:
    # print(zone)
    fig, ax = plt.subplots(1, 1, figsize=(16, 5))
    # Extract the df for this zone
    zone_df = zone_dfs[zone]
    for col in zone_df.columns:
        if isnobal_only:
            if 'ua' or 'nwm' in col:
                continue
        else:
            if not ua_off and nwm_off:
                # if ua_off is False, plot UA and HRRR-SPIReS and SNOW17
                if 'baseline' or 'nwm' in col:
                    continue
            elif not nwm_off and ua_off:
                # if nwm_off is False, plot NWM and HRRR-SPIReS and SNOW17
                if 'baseline' or 'ua' in col:
                    continue
            elif not nwm_off and not ua_off:
                # if both nwm_off and ua_off are False, plot NWM, UA, HRRR-SPIReS and SNOW17
                if 'baseline' in col:
                    continue
            else:
                # only plot HRRR-SPIReS and SNOW17
                if 'baseline' or 'ua' or 'nwm' in col:
                    continue
        color, linestyle, linewidth, alpha = identify_styles(col=col, zone=zone, verbose=False)
        linewidth = linewidth * 3
        zone_df[col].plot(ax=ax, color=color, linestyle=linestyle, linewidth=linewidth, alpha=alpha, label=col)
    ax.set_xlabel('')
    ax.set_ylabel('SWE [m]')
    # Start at the beginning of WY 2022
    ax.set_xlim(pd.to_datetime('2021-10-01'), pd.to_datetime('2024-09-30'))
    ax.set_ylim(0, 1.25)
    ax.grid('lightgrey', linestyle='--', linewidth=1)
    ax.annotate(f'{basin.capitalize()} ({zone})', xy=(0.95, 0.9), xycoords='axes fraction', ha='right', fontsize=22, fontstyle='oblique')
    # Turn off box outline
    plt.legend(bbox_to_anchor=(1.025, 1), loc='upper left', fontsize=18, frameon=False)
    # Save the figure
    plt.savefig(fig_fn, bbox_inches='tight', dpi=300)
    print(fig_fn)

In [ ]:
# Pull out  elevation regions across zones for all models
elev_zones = ['UF', 'MF', 'LF']
elev_dict = dict()
# Loop through elev zones
for elev_zone in elev_zones:
    elev_df_list = []
    # Loop through the abbreviations
    for zone in zones_abbrev:
        # Extract the df for this zone
        zone_df = zone_dfs[zone]
        # Filter the columns to only those contain the elev_zone
        zone_df_huf = zone_df.filter(like=elev_zone)
        elev_df_list.append(zone_df_huf)
    # Concatenate the dataframes for this elev zone
    elev_df = pd.concat(elev_df_list, axis=1, ignore_index=False)
    # Trim to the date range of interest (WY 2022 through WY 2024)
    # shorten by a month for ones that didn't run to completion
    elev_df = elev_df[(elev_df.index >= '2021-10-01') & (elev_df.index <= '2024-08-30')]
    elev_dict[elev_zone] = elev_df

In [ ]:
# Turn it into a dict
elev_big_df = pd.concat(elev_dict, axis=1, keys=elev_dict.keys())
elev_big_df

In [ ]:
# Calculate the standarad deviation by days, this is done down a column
elev_dict[elev_zone].T

In [ ]:
# Update the font size parameters
plt.rcParams.update({'font.size': 15,
                     'axes.labelsize': 18,
                     'axes.titlesize': 16,
                     'xtick.labelsize': 14,
                     'ytick.labelsize': 14,
                     'legend.fontsize': 16})

In [ ]:
stdev_list = []
for idx, elev_zone in enumerate(elev_zones):
    print(idx, elev_zone)
    # Calculate the standard deviation by day
    std_series = elev_dict[elev_zone].T.std().values
    stdev_list.append(std_series)
stdev_df = pd.DataFrame(stdev_list, index=elev_zones).T

# Plot each coluimn
fig, ax = plt.subplots(figsize=(4,3))
sns.boxplot(stdev_df, width=0.2)
ax.set_title(f'{basin.capitalize()} σ in Daily Modeled SWE')
ax.set_ylim(-0.05, 0.5)
ax.grid('lightgray', linestyle='--', linewidth=0.5)
# Annotate with mean, median, n, range and stdev per boxplot
for idx, elev_zone in enumerate(elev_zones):
    mean = stdev_df[elev_zone].mean()
    median = stdev_df[elev_zone].median()
    n = stdev_df[elev_zone].count()
    range_ = stdev_df[elev_zone].max() - stdev_df[elev_zone].min()
    std = stdev_df[elev_zone].std()
    # Add white box behind annotation
    bbox_props = dict(boxstyle='round,pad=0.02', edgecolor='none', facecolor='white', alpha=1)
    ax.annotate(f'{elev_zone}:\nμ={mean:.2f}\nmed={median:.2f}\nn={n}\nrange={range_:.2f}\nσ={std:.2f}',
                xy=(idx, stdev_df[elev_zone].max() + 0.035), ha='center', fontsize=9, bbox=bbox_props)

# Save the figure
fig_fn = PurePath(data_dir, f'basin_zonal_por_figures/{basin}_elevation_zones_swe_stdev_boxplot.png')
plt.savefig(fig_fn, bbox_inches='tight', dpi=300)

In [ ]:
stdev_df.describe()

In [ ]:
# Calculate depletion timing for each model for each zone for each water year and store in a dictionary
sdd_dict = dict()
for zone in zones_abbrev:
    print(zone)
    zone_df = zone_dfs[zone]
    for col in zone_df:
        print(col)
        runtype_elev_zone_series = zone_df[col]
        # Calculate the depletion date for this series for each water year in the record
        for wy in [2022, 2023, 2024]:
            wy_series = runtype_elev_zone_series[(runtype_elev_zone_series.index >= f'{wy-1}-10-01') & (runtype_elev_zone_series.index <= f'{wy}-09-30')]
            if wy_series.isna().sum() > 120:
                print(f'No data for {col} in water year {wy}, skipping...')
                continue
            sdd, _ = proc.calc_sdd(wy_series, alg='first', var='SWE', verbose=False)
            print(sdd)
            sdd_dict[f'{col}_{wy}'] = sdd

sdd_df = pd.DataFrame(sdd_dict, index=['sdd'])
sdd_df

In [ ]:
# Pull out  elevation regions across zones for all models
elev_zones = ['UF', 'MF', 'LF']
sdd_elev_dict = dict()
# Loop through elev zones
for elev_zone in elev_zones:
    sdd_df_elev_sub = sdd_df.filter(like=elev_zone)
    # Loop through WYs
    for wy in [2022, 2023, 2024]:
        sdd_wy_sub = sdd_df_elev_sub.filter(like=f'_{wy}')
        sdd_elev_dict[f'{elev_zone}_{wy}'] = sdd_wy_sub

# Turn it into a dataframe
sdd_big_elev_df = pd.concat(sdd_elev_dict, axis=1, keys=sdd_elev_dict.keys())
sdd_big_elev_df


In [ ]:
# Trasnpose the df
sdd_big_elev_df = sdd_big_elev_df.T
# Turn the indices into columns
sdd_big_elev_df = sdd_big_elev_df.reset_index()
# Rename the columns to be more descriptive
sdd_big_elev_df.columns = ['elev_zone_wy', 'zone_runtype_wy', 'sdd']
# Drop the level_1 (zone_runtype_wy) column
sdd_big_elev_df.drop(columns=['zone_runtype_wy'], inplace=True)
# Set the index to the first column (level_0 or elev_zone_wy)
sdd_big_elev_df.set_index('elev_zone_wy', inplace=True)

# Convert to DOY
sdd_big_elev_df = sdd_big_elev_df.applymap(lambda x: x.dayofyear if pd.notna(x) else np.nan)
# Reset the index into a column again for easy filtering
sdd_big_elev_df.reset_index(inplace=True)
# Break it up into different columns for plotting
sdd_big_df_cols = copy.deepcopy(sdd_big_elev_df)
sdd_big_df_cols[['elev_zone', 'wy']] = sdd_big_df_cols['elev_zone_wy'].str.split('_', expand=True)
# Drop the extraneous elev_zone_wy now
sdd_big_df_cols.drop(columns=['elev_zone_wy'], inplace=True)
sdd_big_df_cols

In [ ]:
# convert doy to yyyymmdd
doy = 130
def doy_to_yyyymmdd(doy, year):
    """Convert day of year to YYYYMMDD string."""
    return pd.to_datetime(f'{year}-01-01') + pd.Timedelta(days=doy - 1)

doy_to_yyyymmdd(doy, 2022)

In [ ]:
sns.boxplot(sdd_big_df_cols, x='elev_zone', y='sdd', hue='wy', width=0.2)

In [ ]:
# Modify elevation zone to ['Upper', 'Middle', 'Lower'] instead of UF, MF, LF for box plotting, but save to new df to not mess up stats
sdd_big_df_cols_long = copy.deepcopy(sdd_big_df_cols)
label_dict = {'UF':'Upper', 'MF':'Middle', 'LF':'Lower'}
sdd_big_df_cols_long['elev_zone'] = sdd_big_df_cols_long['elev_zone'].replace(label_dict)
sdd_big_df_cols_long

In [ ]:
fig, axa = plt.subplots(1, 2, figsize=(12,4), sharey=True)
ax = axa[0]

# Modify palette for zones to have brighter box face colors
palette = ['xkcd:purply', 'xkcd:seaweed green', 'xkcd:golden yellow']

sns.boxplot(sdd_big_df_cols_long, x='wy', y='sdd', hue='elev_zone', ax=ax, width=0.4, palette=palette)
ax.set_xlabel('')

# Reset xticks to prepend WY to each label
ax.set_xticks(np.arange(3))
ax.set_xticklabels([f'WY {wy}' for wy in [2022, 2023, 2024]])
ax.set_ylabel('SWE depletion date')

# Reset ticks to start at April 1
ax.set_yticks(np.arange(91, 305, 30))  # April 1 to Oct 1
ax.set_ylim(60, 304)  # March 1 to Oct 31

# Convert DOY ytick labels Month day with default year start of 2022
ax.set_yticklabels([(pd.to_datetime('2022-01-01') + pd.Timedelta(days=doy-1)).strftime('%b %d') for doy in ax.get_yticks()])
# ax.set_title(f'{basin.capitalize()} SWE depletion timing')
ax.annotate(f'{basin.capitalize()}', xy=(0.95, 0.9), xycoords='axes fraction', ha='right', fontsize=18, fontstyle='oblique')
ax.grid('lightgray', linestyle='--', linewidth=0.5)

# Move legend beyond the frame and spread items full length of y-lims
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize='medium', frameon=False)
# ax.legend(labels=['Upper', 'Middle', 'Lower'], bbox_to_anchor=(1.01, 1), loc='upper left', fontsize='medium', frameon=False)

# Annotate with mean, median, n, range and stdev per zone and water year boxplot (to the right of the ax)
for idx, wy in enumerate([2022, 2023, 2024]):
    for elev_zone in elev_zones:
        this_df = sdd_big_df_cols[(sdd_big_df_cols['elev_zone'] == elev_zone) & (sdd_big_df_cols['wy'] == str(wy))]
        mean = this_df['sdd'].mean()
        median = this_df['sdd'].median()
        n = this_df['sdd'].count()
        range_ = this_df['sdd'].max() - this_df['sdd'].min()
        std = this_df['sdd'].std()
        # Add white box behind annotation
        # print(idx/2 + elev_zones.index(elev_zone)/6, 200)
        print(elev_zones.index(elev_zone)/6, 200)
        bbox_props = dict(boxstyle='round,pad=0.02', edgecolor='none', facecolor='white', alpha=1)
        axa[1].annotate(f'{wy} {elev_zone}:\nμ={mean:.0f}\nmed={median:.0f}\nn={n}\nrange={range_:.0f}\nσ={std:.0f}',
        # axa[1].annotate(f'\nμ={mean:.0f}\nmed={median:.0f}\nn={n}\nrange={range_:.0f}\nσ={std:.0f}',
                    # xy=(idx/2 + elev_zones.index(elev_zone)/6, 200),
                    # xy=(0.1 + elev_zones.index(elev_zone)/5, 240 - idx*90),
                    xy=(0.3 + idx/4, 240 - elev_zones.index(elev_zone) * 90),
                    ha='center', fontsize=11, bbox=bbox_props)
# Turn off the second axis frame
axa[1].axis('off')

# Save the figure
fig_fn = PurePath(data_dir, f'basin_zonal_por_figures/{basin}_elevation_zones_wy_sdd_boxplot.png')
plt.savefig(fig_fn, bbox_inches='tight', dpi=300)

In [ ]:
fig, axa = plt.subplots(1, 2, figsize=(12,4), sharey=True)
ax = axa[0]

# Modify palette for zones to have brighter box face colors
palette = ['xkcd:purply', 'xkcd:seaweed green', 'xkcd:golden yellow']

sns.boxplot(sdd_big_df_cols_long, x='wy', y='sdd', hue='elev_zone', ax=ax, width=0.4, palette=palette)
ax.set_xlabel('')

# Reset xticks to prepend WY to each label
ax.set_xticks(np.arange(3))
ax.set_xticklabels([f'WY {wy}' for wy in [2022, 2023, 2024]])
ax.set_ylabel('SWE depletion date')

# Reset ticks to start at April 1
ax.set_yticks(np.arange(91, 305, 30))  # April 1 to Oct 1
ax.set_ylim(60, 304)  # March 1 to Oct 31

# Convert DOY ytick labels Month day with default year start of 2022
ax.set_yticklabels([(pd.to_datetime('2022-01-01') + pd.Timedelta(days=doy-1)).strftime('%b %d') for doy in ax.get_yticks()])
# ax.set_title(f'{basin.capitalize()} SWE depletion timing')
ax.annotate(f'{basin.capitalize()}', xy=(0.95, 0.9), xycoords='axes fraction', ha='right', fontsize=18, fontstyle='oblique')
ax.grid('lightgray', linestyle='--', linewidth=0.5)

# Move legend beyond the frame and spread items full length of y-lims
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize='medium', frameon=False)
# ax.legend(labels=['Upper', 'Middle', 'Lower'], bbox_to_anchor=(1.01, 1), loc='upper left', fontsize='medium', frameon=False)

# Annotate with mean, median, n, range and stdev per zone and water year boxplot (to the right of the ax)
for idx, wy in enumerate([2022, 2023, 2024]):
    for elev_zone in elev_zones:
        this_df = sdd_big_df_cols[(sdd_big_df_cols['elev_zone'] == elev_zone) & (sdd_big_df_cols['wy'] == str(wy))]
        mean = this_df['sdd'].mean()
        median = this_df['sdd'].median()
        n = this_df['sdd'].count()
        range_ = this_df['sdd'].max() - this_df['sdd'].min()
        iqr = np.nanpercentile(this_df['sdd'], 75) - np.nanpercentile(this_df['sdd'], 25)
        # Add white box behind annotation
        # print(idx/2 + elev_zones.index(elev_zone)/6, 200)
        print(elev_zones.index(elev_zone)/6, 200)
        bbox_props = dict(boxstyle='round,pad=0.02', edgecolor='none', facecolor='white', alpha=1)
        axa[1].annotate(f'{wy} {elev_zone}:\nμ={mean:.0f}\nmed={median:.0f}\nn={n}\nrange={range_:.0f}\niqr={iqr:.0f}',
                    xy=(0.3 + idx/4, 240 - elev_zones.index(elev_zone) * 90),
                    ha='center', fontsize=11, bbox=bbox_props)
# Turn off the second axis frame
axa[1].axis('off')

# Save the figure
fig_fn = PurePath(data_dir, f'basin_zonal_por_figures/{basin}_elevation_zones_wy_sdd_boxplot.png')
plt.savefig(fig_fn, bbox_inches='tight', dpi=300)

In [ ]:
# now reconstruct the dataset so that each column is the elev_zone_wy and
# the rows are all the corresponding sdd vlaues for ease of stat calcs
sdd_big_df = pd.DataFrame()
for elev_zone_wy in sdd_big_elev_df['elev_zone_wy'].unique():
    # Filter the dataframe to only this elev_zone_wy
    sub_df = sdd_big_elev_df[sdd_big_elev_df['elev_zone_wy'] == elev_zone_wy]
    # Drop the elev_zone_wy column
    sub_df.drop(columns=['elev_zone_wy'], inplace=True)
    # Rename the sdd column to the elev_zone_wy
    sub_df.columns = [elev_zone_wy]
    # And reset the index inplace
    sub_df.reset_index(drop=True, inplace=True)
    # Concatenate to the final dataframe
    sdd_big_df = pd.concat([sdd_big_df, sub_df], axis=1)
# sdd_big_df

# Now create an index of elev zones for the columns, grouping into UF, MF, and LF
sdd_big_df_elev = copy.deepcopy(sdd_big_df)
sdd_big_df_elev.columns = sdd_big_df_elev.columns.str.split('_', expand=True)
sdd_big_df_elev.columns.names = ['elev_zone', 'water_year']
# sdd_big_df_elev
sdd_big_df_elev.describe()

In [ ]:
zone_df

In [ ]:
# Now group by specific site
# Calculate depletion timing for each model for each zone for each water year and store in a dictionary
zone_sdd_dict = dict()
for zone in zones_abbrev:
    print(zone)
    zone_df = zone_dfs[zone]
    for col in zone_df:
        runtype_elev_zone_series = zone_df[col]
        # Calculate the depletion date for this series for each water year in the record
        for wy in [2022, 2023, 2024]:
            wy_series = runtype_elev_zone_series[(runtype_elev_zone_series.index >= f'{wy-1}-10-01') & (runtype_elev_zone_series.index <= f'{wy}-09-30')]
            if wy_series.isna().sum() > 120:
                print(f'No data for {col} in water year {wy}, skipping...')
                continue
            sdd, _ = proc.calc_sdd(wy_series, alg='first', var='SWE', verbose=False)
            print(sdd)
            zone_sdd_dict[f'{col}_{wy}'] = sdd

# sdd_df = pd.DataFrame(sdd_dict, index=['sdd'])
# sdd_df

In [ ]:
# Turn it into a df
zone_sdd_df = pd.DataFrame(zone_sdd_dict, index=['sdd']).T
# Convert index to columns
zone_sdd_df.reset_index(inplace=True)
# Generate new columns from the index column
zone_sdd_df[['zone', 'runtype', 'water_year']] = zone_sdd_df['index'].str.split('_', expand=True)
# Drop the index column
zone_sdd_df.drop(columns=['index'], inplace=True)
# Group by zone
zone_sdd_df = zone_sdd_df.groupby(['zone', 'runtype', 'water_year']).mean().reset_index()
# Turn the zone into a column index
zone_sdd_df = zone_sdd_df.pivot_table(index='runtype', columns=['zone', 'water_year'], values='sdd')
zone_sdd_df

In [ ]:
# Statify it
zone_sdd_df.describe()

In [ ]:
# Try to coerce to doy
zone_sdd_df = zone_sdd_df.applymap(lambda x: x.dayofyear if pd.notna(x) else np.nan)
zone_sdd_df

In [ ]:
# Only show to tenths decimal place
zone_sdd_df.describe().applymap(lambda x: f"{x:0.1f}")

In [ ]:
# Locate the zone that has the lowest variabiliy between models --> in terms of standard deviation
zone_sdd_df.std().sort_values(ascending=True)

In [ ]:
# # Pull out the SKEC2HLF zone data
# zone_sdd_df['SKEC2HLF'].describe().applymap(lambda x: f"{x:0.1f}")

In [ ]:
# Pull out the BUEC2HLF zone data
zone_sdd_df['BUEC2HLF'].describe().applymap(lambda x: f"{x:0.1f}")